# World-generation time under WorldGeneration

A self-contained experiment that runs the `WorldGeneration` workload against a
freshly-started Minecraft server and plots the time each emulated player takes
to complete its teleport/load cycles.

Each player joins, switches to spectator mode, and then teleports to a fresh
location far from spawn and far from the other players, waits for that area's
chunks to load in (forcing the server to generate them), and immediately
teleports again -- 32 times. The per-player total time and per-teleport load
latency are written **directly to InfluxDB** by the workload (the
`minecraft_worldgen` and `minecraft_worldgen_teleport` measurements); no
Telegraf is involved for these points.

**Configuration**

| Component | Setting |
|-----------|---------|
| Game | `MinecraftServer` (itzg/minecraft-server:java25, RCON enabled) |
| Workload | `WorldGeneration1User` and `WorldGeneration8Users` |
| Teleports per player | 32 |
| Metrics path | workload -> InfluxDB HTTP write API |

**Sweep variable:** player count (1 vs 8), one group per value.

## Setup

Imports and experiment parameters.

In [ ]:
# Auto-reload edited yardstick_benchmark modules without restarting the
# kernel (handy while iterating on the workload code). Note: a module already
# imported in a running kernel BEFORE its source changed still needs one
# kernel restart; autoreload only tracks edits made after the kernel started.
%load_ext autoreload
%autoreload 2

import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt

import yardstick_benchmark
from yardstick_benchmark.games.minecraft.server import (
    MinecraftServer,
    MinecraftServerCrashed,
)
from yardstick_benchmark.games.minecraft.workload import WorldGeneration
from yardstick_benchmark.model import Node
from yardstick_benchmark.monitoring import InfluxDB
from yardstick_benchmark.util import wait_for_url

from _lib import query_influxdb_dataframe

# Our Flux queries intentionally return long-format, single-field data and
# don't pivot(); silence influxdb-client's benign performance hint about it
# so it doesn't flood the run cell's output.
from influxdb_client.client.warnings import MissingPivotFunction
warnings.simplefilter("ignore", MissingPivotFunction)

NODE = Node("localhost", Path("/tmp/ysat-worldgen"))
# Pin to a Minecraft version the workload image's Mineflayer supports
# (itzg's LATEST outpaces minecraft-data, so the bots can't connect).
MC_VERSION = MinecraftServer.DEFAULT_VERSION
TELEPORTS = 32

## Run the experiment

Deploy + start the stack, run the 1-user and 8-user workloads back to back,
query the committed metric, then tear down. The DataFrames are captured before
teardown so the plot cells can be re-run independently.

In [ ]:
yardstick_benchmark.clean([NODE])

influxdb = InfluxDB(NODE)
minecraft = MinecraftServer("yardstick-worldgen-mc", version=MC_VERSION)

summary = None
teleport = None
try:
    influxdb.deploy()
    influxdb.start()
    wait_for_url(f"{influxdb.url}/health", timeout_s=60)

    minecraft.start()
    minecraft.wait_until_ready()
    minecraft.set_world_spawn(0, 0)
    # MC runs as a background-service instance; poll its log in the
    # background so a crash is caught whenever it happens.
    minecraft.start_health_monitor()

    info = influxdb.get_info()
    # Scope the result queries to this run so points from earlier runs in the
    # bucket don't leak into the plots.
    run_start = datetime.now(timezone.utc)
    # If the server crashes mid-run we abort the remaining workloads but still
    # keep whatever partial data already landed in InfluxDB (so the cell
    # doesn't error out and the plots can render what we got).
    crashed = None
    try:
        for bots in (1, 4):
            print(f"running WorldGeneration ({bots} player(s))...")
            workload = WorldGeneration(NODE, "localhost", info,
                                       minecraft.rcon_password,
                                       bots_per_node=bots,
                                       minecraft_version=MC_VERSION)
            workload.deploy()
            try:
                # Foreground `apptainer run`: blocks until the workload exits
                # (streaming its output), aborting early (raising
                # MinecraftServerCrashed) if the background health monitor
                # flags a server crash mid-run.
                workload.run(health_check=minecraft.assert_healthy)
                print(f"  {bots}-player run done.")
            finally:
                workload.cleanup()
    except MinecraftServerCrashed as e:
        crashed = e
        print(f"!! run aborted: Minecraft server crashed: {e}".splitlines()[0])
        print("!! keeping partial data collected before the crash.")

    start = run_start.isoformat()
    # Per-player total time (one row per player per run).
    summary = query_influxdb_dataframe(
        influxdb,
        f'''
from(bucket: "yardstick")
  |> range(start: {start})
  |> filter(fn: (r) => r._measurement == "minecraft_worldgen" and r._field == "total_duration_ms")
  |> keep(columns: ["_time", "_value", "player", "bots"])
  |> group()
''',
    )
    # Per-teleport load latency.
    teleport = query_influxdb_dataframe(
        influxdb,
        f'''
from(bucket: "yardstick")
  |> range(start: {start})
  |> filter(fn: (r) => r._measurement == "minecraft_worldgen_teleport" and r._field == "load_ms")
  |> keep(columns: ["_time", "_value", "player", "bots", "teleport"])
  |> group()
''',
    )
finally:
    # Defensive teardown: stop every component we may have started, even on
    # KeyboardInterrupt, and don't let a not-yet-started component's
    # "no instance found" error mask the original failure/interrupt.
    minecraft.stop_health_monitor()
    for name, component in (("minecraft", minecraft), ("influxdb", influxdb)):
        try:
            component.stop()
        except Exception as e:
            print(f"warning: {name}.stop() failed: {e}")

print(f"collected {len(summary)} player summaries, {len(teleport)} teleport samples")
summary

## Plot per-player total world-generation time

The time each player took to complete its 32 teleport/load cycles, grouped by
run size (1 vs 8 players).

In [ ]:
import seaborn as sns

sns.set_theme(style="ticks", context="notebook", font="DejaVu Sans")

summary_plot = summary.copy()
summary_plot["total_s"] = summary_plot["_value"] / 1000.0

fig, ax = plt.subplots(figsize=(7, 4))
sns.stripplot(data=summary_plot, x="bots", y="total_s", ax=ax, size=8, jitter=0.15)
ax.set_xlabel("Players in run")
ax.set_ylabel(f"Time for {TELEPORTS} teleports [s]")
ax.set_title("World-generation time per player")
ax.set_ylim(bottom=0)
sns.despine()
plt.show()

## Plot per-teleport load latency

How long each individual teleport took to load in, by teleport index. Later
teleports always reach further-out, freshly generated terrain.

In [ ]:
teleport_plot = teleport.copy()
teleport_plot["teleport"] = teleport_plot["teleport"].astype(int)
teleport_plot["load_s"] = teleport_plot["_value"] / 1000.0

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(
    data=teleport_plot,
    x="teleport",
    y="load_s",
    hue="bots",
    errorbar="sd",
    marker="o",
    ax=ax,
)
ax.set_xlabel("Teleport index")
ax.set_ylabel("Chunk load time [s]")
ax.set_title("Per-teleport world-generation latency")
ax.set_ylim(bottom=0)
ax.legend(title="players")
sns.despine()
plt.show()